[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/03_tag_11_12_ar_zu_arma.ipynb)

# Tag 11 & 12 – 03: AR und ARMA trainieren und vergleichen

Dieses Beispiel trainiert lineare **AR-** und **ARMA-Modelle** auf einem frühen Abschnitt einer Zeitreihe und bewertet sie anschließend auf einem späteren, unbekannten Abschnitt. Es gibt keine interaktiven Regler und keine von Hand eingestellten Modellgewichte.

## Was wird hier gelernt?

Ein AR-Modell lernt Gewichte für die letzten $p$ Messwerte. Ein ARMA-Modell lernt zusätzlich Gewichte für die letzten $q$ **AR-Prognosefehler**. Die Fehler werden in einer Walk-forward-Sicht verwendet: Sobald ein tatsächlicher Wert eingetroffen ist, ist sein Fehler bekannt und darf für die nächste Vorhersage genutzt werden.

Die Gewichtsschätzung erfolgt mit der Methode der kleinsten Quadrate – derselben Grundidee wie bei einer linearen Regression. Das Verfahren ist hier bewusst direkt mit NumPy umgesetzt, damit sichtbar bleibt, welche Eingaben das Modell erhält.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 21

## Konfiguration für den Vergleich

Die folgenden Werte sind die einzige Stelle, die für eigene Vergleiche geändert werden muss. `p` beschreibt die Länge des Messwert-Fensters; `q` beschreibt, wie viele vergangene Fehler zusätzlich verwendet werden. `q=0` bedeutet: reines AR-Modell.

In [ ]:
# Zum Vergleichen ändern:
TRAINING_ANTEIL = 70
VERGLEICHE = [(4, 0), (4, 2), (12, 0), (12, 2), (24, 0), (24, 2), (36, 0), (36, 2)]  # (p, q)
DETAIL_MODELL = (24, 2)  # dieses trainierte Modell wird im Detail gezeigt
DETAIL_ZEITPUNKT = 600  # ein Zeitpunkt im Testbereich

assert DETAIL_MODELL in VERGLEICHE
assert 0 < TRAINING_ANTEIL < 100

## Künstliche periodische Zeitreihe

Die Reihe enthält einen klaren Hauptzyklus von 24 Zeitschritten und einen kleineren, schnelleren Zyklus. Das ist vergleichbar mit einem täglichen Verlauf mit wiederkehrenden Schwankungen innerhalb eines Tages. Kleine, zeitlich korrelierte Abweichungen bleiben bewusst enthalten: Die AR-Modelle können den periodischen Verlauf lernen; der MA-Anteil kann zusätzlich nachwirkende Abweichungen berücksichtigen.

In [ ]:
def erzeuge_zeitreihe(anzahl_werte=720, rauschen=0.14, seed=RANDOM_STATE):
    """Erzeugt einen periodischen Verlauf mit einem echten Moving-Average-Störanteil."""
    rng = np.random.default_rng(seed)
    zeit = np.arange(anzahl_werte)
    periodischer_verlauf = (
        1.15 * np.sin(2 * np.pi * zeit / 24)
        + 0.35 * np.sin(2 * np.pi * zeit / 8 + 0.7)
    )
    zufall = rng.normal(0, rauschen, anzahl_werte)
    # Ein einmaliger Zufallsschock beeinflusst die nächsten zwei Werte mit.
    nachwirkende_abweichung = (
        zufall
        + 2.00 * np.r_[0.0, zufall[:-1]]
        - 0.80 * np.r_[0.0, 0.0, zufall[:-2]]
    )
    werte = periodischer_verlauf + nachwirkende_abweichung
    return werte


werte = erzeuge_zeitreihe()
trennindex = int(len(werte) * TRAINING_ANTEIL / 100)

plt.figure(figsize=(12, 3.5))
plt.plot(werte, color='0.35', linewidth=1.4)
plt.axvspan(0, trennindex - 1, color='tab:blue', alpha=0.10, label='Training: Modelle sehen diese Werte')
plt.axvspan(trennindex, len(werte) - 1, color='tab:orange', alpha=0.10, label='Test: erst danach bewerten')
plt.axvline(trennindex - 0.5, color='black', linestyle='--', linewidth=1)
plt.title('Chronologische Trennung der Zeitreihe')
plt.xlabel('Zeitindex')
plt.ylabel('Messwert')
plt.legend(loc='upper right')
plt.show()

print(f'Gesamt: {len(werte)} Werte | Training: {trennindex} | Test: {len(werte) - trennindex}')

## AR und ARMA aus dem Trainingsabschnitt schätzen

Die Funktion trainiert zuerst ein AR-Modell mit den vergangenen Messwerten. Dessen Residuen werden danach als zusätzliche Eingaben für das ARMA-Modell verwendet. Für den Testbereich werden Vorhersagen nacheinander erzeugt: Ein Residuum kann immer erst ab dem folgenden Zeitpunkt verwendet werden.

In [ ]:
def fit_lineares_modell(X, y):
    """Schätzt Achsenabschnitt und Gewichte durch kleinste Quadrate."""
    X_mit_bias = np.column_stack([np.ones(len(X)), X])
    parameter, *_ = np.linalg.lstsq(X_mit_bias, y, rcond=None)
    return parameter[0], parameter[1:]


def vorhersage(X, bias, gewichte):
    return bias + np.asarray(X) @ gewichte


def trainiere_und_teste(reihe, ordnung_p, ma_ordnung_q, training_anteil):
    """Trainiert auf dem frühen Teil und erzeugt Walk-forward-Prognosen im späteren Testteil."""
    trennindex = int(len(reihe) * training_anteil / 100)

    # AR-Training: die letzten p Werte sagen den nächsten Wert vorher.
    X_ar = np.array([reihe[t - ordnung_p : t] for t in range(ordnung_p, trennindex)])
    y_ar = reihe[ordnung_p:trennindex]
    ar_bias, ar_gewichte = fit_lineares_modell(X_ar, y_ar)

    # Ein-Schritt-AR-Prognosen und daraus entstehende Residuen.
    ar_alle = np.full(len(reihe), np.nan)
    for t in range(ordnung_p, len(reihe)):
        ar_alle[t] = vorhersage(reihe[t - ordnung_p : t], ar_bias, ar_gewichte)
    ar_residuen = reihe - ar_alle

    if ma_ordnung_q == 0:
        return {
            'p': ordnung_p, 'q': 0, 'trennindex': trennindex,
            'bias': ar_bias, 'werte_gewichte': ar_gewichte, 'fehler_gewichte': np.array([]),
            'ar_test': ar_alle, 'arma_test': ar_alle, 'ar_residuen': ar_residuen,
        }

    # ARMA-Training: p Messwerte und q vorherige AR-Residuen sind die Eingaben.
    erster_arma_zeitpunkt = ordnung_p + ma_ordnung_q
    X_arma = np.array([
        np.concatenate([reihe[t - ordnung_p : t], ar_residuen[t - ma_ordnung_q : t]])
        for t in range(erster_arma_zeitpunkt, trennindex)
    ])
    y_arma = reihe[erster_arma_zeitpunkt:trennindex]
    arma_bias, arma_gewichte = fit_lineares_modell(X_arma, y_arma)

    # Walk-forward-Test: Es werden nur Werte und Fehler von Zeitpunkten vor t eingesetzt.
    arma_test = np.full(len(reihe), np.nan)
    for t in range(trennindex, len(reihe)):
        vergangene_werte = reihe[t - ordnung_p : t]
        vergangene_fehler = ar_residuen[t - ma_ordnung_q : t]
        eingabe = np.concatenate([vergangene_werte, vergangene_fehler])
        arma_test[t] = vorhersage(eingabe, arma_bias, arma_gewichte)

    return {
        'p': ordnung_p, 'q': ma_ordnung_q, 'trennindex': trennindex,
        'bias': arma_bias, 'werte_gewichte': arma_gewichte[:ordnung_p],
        'fehler_gewichte': arma_gewichte[ordnung_p:],
        'ar_test': ar_alle, 'arma_test': arma_test, 'ar_residuen': ar_residuen,
    }

## Mehrere Fensterlängen vergleichen

Jede Zeile wird unabhängig neu trainiert. Der MAE (mittlerer absoluter Fehler) wird nur auf dem Testbereich berechnet: kleiner ist besser. So lässt sich sehen, ob ein längeres Messwert-Fenster oder zusätzliche Fehlerwerte in dieser Zeitreihe hilfreich sind.

In [ ]:
ergebnisse = []
modelle = {}
for p, q in VERGLEICHE:
    modell = trainiere_und_teste(werte, p, q, TRAINING_ANTEIL)
    test_start = modell['trennindex']
    prognose = modell['arma_test'][test_start:]
    istwerte = werte[test_start:]
    mae = np.nanmean(np.abs(istwerte - prognose))
    bezeichnung = f'AR({p})' if q == 0 else f'ARMA({p}, {q})'
    ergebnisse.append((bezeichnung, p, q, mae))
    modelle[(p, q)] = modell

print(f'{"Modell":<12} {"p":>3} {"q":>3} {"Test-MAE":>10}')
print('-' * 33)
for bezeichnung, p, q, mae in ergebnisse:
    print(f'{bezeichnung:<12} {p:>3} {q:>3} {mae:>10.3f}')

bezeichnungen = [ergebnis[0] for ergebnis in ergebnisse]
maes = [ergebnis[3] for ergebnis in ergebnisse]
farben = ['tab:orange' if q == 0 else 'tab:purple' for _, _, q, _ in ergebnisse]
plt.figure(figsize=(10, 4))
balken = plt.bar(bezeichnungen, maes, color=farben)
plt.bar_label(balken, fmt='%.3f', padding=3)
plt.title('Testfehler verschiedener trainierter Modelle')
plt.ylabel('MAE – kleiner ist besser')
plt.ylim(0, max(maes) * 1.22)
plt.show()

print('Orange: AR ohne Fehlerwerte | Violett: ARMA mit vergangenen Fehlern')

## Ein trainiertes AR-Modell gegen ein trainiertes ARMA-Modell

Für das ausgewählte Modell aus `DETAIL_MODELL` zeigt die Grafik die Ein-Schritt-Prognosen im Testbereich. Beide Varianten verwenden die gleiche Messwert-Vergangenheit. Das ARMA-Modell darf zusätzlich die letzten Fehler verwenden.

In [ ]:
p, q = DETAIL_MODELL
detail = modelle[DETAIL_MODELL]
test_start = detail['trennindex']
test_x = np.arange(test_start, len(werte))
ar_mae = np.nanmean(np.abs(werte[test_start:] - detail['ar_test'][test_start:]))
arma_mae = np.nanmean(np.abs(werte[test_start:] - detail['arma_test'][test_start:]))

plt.figure(figsize=(12, 4.5))
plt.plot(test_x, werte[test_start:], color='0.25', linewidth=1.8, label='tatsächliche Testwerte', zorder=1)
plt.plot(test_x, detail['arma_test'][test_start:], color='tab:purple', linestyle='--', linewidth=2.2, alpha=0.85, label=f'trainiertes ARMA({p}, {q})', zorder=2)
plt.plot(test_x, detail['ar_test'][test_start:], color='tab:orange', linestyle='-', linewidth=1.5, marker='o', markersize=2.8, markevery=4, label=f'trainiertes AR({p})', zorder=3)
plt.title(f'Ein-Schritt-Prognosen im Testbereich | MAE AR={ar_mae:.3f}, ARMA={arma_mae:.3f}')
plt.xlabel('Zeitindex')
plt.ylabel('Messwert')
plt.legend(loc='upper right')
plt.show()

print('Gelernte Gewichte für vergangene Messwerte:', np.round(detail['werte_gewichte'], 3))
print('Gelernte Gewichte für vergangene Fehler:', np.round(detail['fehler_gewichte'], 3))

## Was nutzt die konkrete Vorhersage?

Der folgende Ausschnitt zeigt einen festen Zeitpunkt aus dem Testbereich. Die blauen Punkte sind die letzten $p$ tatsächlich beobachteten Messwerte; sie gehen in **beide** Modelle ein. Die Balken darunter zeigen die letzten $q$ AR-Residuen, die nur das ARMA-Modell zusätzlich verwenden darf.

In [ ]:
t = max(test_start, min(DETAIL_ZEITPUNKT, len(werte) - 1))
werte_indizes = np.arange(t - p, t)
fehler_indizes = np.arange(t - q, t)
aktive_werte = werte[werte_indizes]
aktive_fehler = detail['ar_residuen'][fehler_indizes]

fig, (ax_zeitreihe, ax_fehler) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [2, 1]})
fenster_start = max(0, t - p - 12)
fenster_ende = min(len(werte), t + 8)
ausschnitt = np.arange(fenster_start, fenster_ende)
ax_zeitreihe.plot(ausschnitt, werte[ausschnitt], 'o-', color='0.55', markersize=4, label='Zeitreihe')
ax_zeitreihe.plot(werte_indizes, aktive_werte, 'o-', color='tab:blue', linewidth=3, markersize=7, label=f'aktive Eingabe: letzte p={p} Werte')
ax_zeitreihe.scatter(t, werte[t], s=110, color='tab:green', zorder=4, label='tatsächlicher Wert')
ax_zeitreihe.scatter(t, detail['ar_test'][t], s=100, color='tab:orange', zorder=4, label='AR-Prognose')
ax_zeitreihe.scatter(t, detail['arma_test'][t], s=100, color='tab:purple', zorder=4, label='ARMA-Prognose')
ax_zeitreihe.set(title=f'Aktive Vorhersage bei t={t}', xlabel='Zeitindex', ylabel='Messwert')
ax_zeitreihe.legend(ncol=3, loc='lower left')

fehler_beitraege = detail['fehler_gewichte'] * aktive_fehler
labels = [f'e(t-{t - i})' for i in fehler_indizes]
farben = ['tab:blue' if beitrag >= 0 else 'crimson' for beitrag in fehler_beitraege]
ax_fehler.bar(labels, fehler_beitraege, color=farben, alpha=0.85)
ax_fehler.axhline(0, color='0.3', linewidth=0.8)
ax_fehler.set(title=f'Zusätzliche ARMA-Eingaben: letzte q={q} bekannte AR-Residuen', ylabel='gelerntes Gewicht × Fehler')
for label, beitrag, fehler, gewicht in zip(labels, fehler_beitraege, aktive_fehler, detail['fehler_gewichte']):
    ax_fehler.text(label, beitrag, f'e={fehler:.2f}\nw={gewicht:.2f}', ha='center', va='bottom' if beitrag >= 0 else 'top')
plt.tight_layout()
plt.show()

print(f'AR nutzt bei t={t} die Messwerte an den Indizes {werte_indizes.tolist()}.')
print(f'ARMA nutzt zusätzlich die bekannten AR-Residuen an den Indizes {fehler_indizes.tolist()}.')
print(f'AR-Prognose: {detail["ar_test"][t]:.3f} | ARMA-Prognose: {detail["arma_test"][t]:.3f} | Istwert: {werte[t]:.3f}')

## Kerngedanke

Die Sequenzlänge $p$ bestimmt, wie viele Messwerte eine Prognose sehen darf. Die MA-Ordnung $q$ bestimmt, wie viele vergangene Fehler hinzukommen. Mehr Werte sind nicht automatisch besser: Ein zu langes Fenster kann unnötige Parameter hinzufügen. Deshalb werden mehrere Kombinationen trainiert und ausschließlich auf späteren Testwerten verglichen.